In [2]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# load dataset
file_path = "../data/raw/i80_trajectories_NGSIM.csv"

df = pd.read_csv(file_path)

df.head()

C:\Users\adama\AppData\Local\Temp\ipykernel_29880\34955442.py:4: DtypeWarning: Columns (0: Local_Y, 1: Space_Headway) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


,Vehicle_ID,Frame_ID,Total_Frames,Global_Time,Local_X,Local_Y,Global_X,Global_Y,v_length,v_Width,...,D_Zone,Int_ID,Section_ID,Direction,Movement,Preceding,Following,Space_Headway,Time_Headway,Location
0,2071,8323,379,1113438397200,7.224,0.0,6042842.456,2133039.613,12.3,6.8,...,NaN,NaN,NaN,NaN,NaN,"2,060",0,146.57,"9,999.99",i-80
1,2071,8324,379,1113438397300,7.239,0.0,6042842.112,2133042.392,12.3,6.8,...,NaN,NaN,NaN,NaN,NaN,"2,060",0,150.25,"9,999.99",i-80
2,1895,6565,"1,958",1113438221400,42.224,0.0,6042877.191,2133043.912,14.8,6.9,...,NaN,NaN,NaN,NaN,NaN,"1,887",0,40.08,"9,999.99",i-80
3,1895,6566,"1,958",1113438221500,42.227,0.0,6042877.124,2133044.458,14.8,6.9,...,NaN,NaN,NaN,NaN,NaN,"1,887",0,41.1,"9,999.99",i-80
4,1895,6567,"1,958",1113438221600,42.230,0.0,6042877.056,2133045.004,14.8,6.9,...,NaN,NaN,NaN,NaN,NaN,"1,887",0,42.16,"9,999.99",i-80


In [4]:
# compute trajectory lengths
trajectory_lengths = df.groupby('Vehicle_ID').size()

In [5]:
# filter vehicles with enough frames
valid_vehicles = trajectory_lengths[trajectory_lengths >= 30].index

df_filtered = df[df['Vehicle_ID'].isin(valid_vehicles)]

In [6]:
# check size after filtering
df_filtered.shape

(4566387, 25)

In [7]:
# downsample (10 Hz → 5 Hz)
df_downsampled = df_filtered[df_filtered['Frame_ID'] % 2 == 0]

In [8]:
# check size
df_downsampled.shape

(2283186, 25)

## Sliding Window ANN Input

In [9]:
# sort by vehicle and time
df_downsampled = df_downsampled.sort_values(by=['Vehicle_ID', 'Frame_ID'])

In [10]:
# select features
features = ['Local_X', 'Local_Y', 'v_Vel', 'v_Acc', 'Lane_ID']

In [11]:
# storage for dataset
X = []  # inputs
y = []  # labels

In [12]:
# group by vehicle
grouped = df_downsampled.groupby('Vehicle_ID')
# check number of groups
len(grouped)

3001

In [13]:
# window size
window_size = 15

In [14]:
# build input windows
for vehicle_id, group in grouped:
    
    group = group.sort_values(by='Frame_ID')
    
    data = group[features].values
    
    for i in range(len(data) - window_size):
        
        window = data[i:i+window_size]
        
        X.append(window.flatten())

In [15]:
# number of samples
len(X)

2238171

In [16]:
# shape of one sample
len(X[0])

75

In [17]:
# reset storage
X = []
y = []

In [18]:
# parameters
window_size = 15
future_horizon = 15

In [19]:
# build windows with labels
for vehicle_id, group in grouped:
    
    group = group.sort_values(by='Frame_ID')
    
    data = group[features].values
    lanes = group['Lane_ID'].values
    
    for i in range(len(data) - window_size - future_horizon):
        
        # input window
        window = data[i:i+window_size]
        
        # future lane values
        future = lanes[i+window_size : i+window_size+future_horizon]
        
        # label: 1 if lane changes in future
        label = int(np.any(future != lanes[i+window_size-1]))
        
        X.append(window.flatten())
        y.append(label)

In [20]:
# number of samples
len(X), len(y)

(2193156, 2193156)

In [21]:
# check label distribution
np.unique(y, return_counts=True)

(array([0, 1]), array([1785931,  407225]))

In [22]:
# clean and convert X to numeric
X_clean = np.array([
    [float(str(value).replace(',', '')) for value in row]
    for row in X
], dtype=np.float32)

# convert y
y_clean = np.array(y, dtype=np.int64)

In [23]:
# save clean dataset
np.save("../data/processed/X.npy", X_clean)
np.save("../data/processed/y.npy", y_clean)